## Model 0: baseline dummy classifier

First, let us download the data set from Kaggle (if we haven't already done so):

In [1]:
import os
import shutil

import kagglehub
import numpy as np
import pandas as pd

if not os.path.isdir("nlp-getting-started"):
    path = kagglehub.competition_download("nlp-getting-started")
    shutil.move(path, "nlp-getting-started")

df_train = pd.read_csv("nlp-getting-started/train.csv")
df_test = pd.read_csv("nlp-getting-started/test.csv")

X_train = df_train.drop(columns=["target"])
y_train = df_train["target"]
X_test = df_test # No corresponding target column

Now we can see the data:

In [2]:
df_train

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [3]:
df_test

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan
...,...,...,...,...
3258,10861,NaN,NaN,EARTHQUAKE SAFETY LOS ANGELES ÛÒ SAFETY FASTE...
3259,10865,NaN,NaN,Storm in RI worse than last hurricane. My city...
3260,10868,NaN,NaN,Green Line derailment in Chicago http://t.co/U...
3261,10874,NaN,NaN,MEG issues Hazardous Weather Outlook (HWO) htt...


Now that that's done, let's create a dummy classifier with the default strategy for this data:

In [4]:
from sklearn.dummy import DummyClassifier

classifier = DummyClassifier()
classifier.fit(X_train, y_train)
y_test = classifier.predict(df_test)

df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
df_result.set_index("id", inplace=True)
df_result.to_csv("0_dummy.csv")

Uploading this to Kaggle gives us a score of 57.033% - approximately a coin-flip. This would indicate that the test data is fairly balanced. And if we analyze the training data, we find that it is as well.

In [5]:
y_train.value_counts()

target
0    4342
1    3271
Name: count, dtype: int64

With that in mind, there will be no need to use F1 scores or other complicated metrics when performing cross validation or grid searches later on.

## Model 1: Simple bag-of-words with Random Forest

Now, let's train an actual model on this data. First, let's create a bag-of-words vectorizer for the tweets. By default, the Scikit-Learn tokenizer for `CountVectorizer` is rather simplistic, and performs no stemming nor is it aware of things like URLs. To that end, we'll create a custom one based on the NLTK `SnowballStemmer` and `TweetTokenizer`:

In [6]:
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("stopwords") # ICYNI

STEMMER = SnowballStemmer(language = "english")
TOKENIZER = TweetTokenizer(preserve_case=False, reduce_len=True, strip_handles=True)
URL_RE = re.compile(r"^(https?://|www\.)")
HAS_AZ_RE = re.compile(r"[a-z]")

def stem_tokenize(text):
    tokens = TOKENIZER.tokenize(text)
    stems = []
    
    for tok in tokens:
        #remove urls
        if URL_RE.match(tok):
            continue

        # This removes the #, I am not sure if we want to remove so the words in # get tokenized like normal,
        # or if we want to some way signify that there is a # in there
        tok = tok.lstrip("#")

        # must have char a-z
        if not HAS_AZ_RE.search(tok):
            continue

        stems.append(STEMMER.stem(tok))

    return stems

base_stop_words = set(stopwords.words("english"))

keep_words = {
    'not', 'no', 'nor',
    'up', 'down', 'out', 'off',
    'after', 'before', 'during',
    'again', 'more', 'all',
    'over', 'only'
}
base_stop_words -= keep_words

STOP_WORDS = sorted({stop_w for words in base_stop_words for stop_w in stem_tokenize(words)})

bow_vectorizer = CountVectorizer(
    tokenizer=stem_tokenize,
    token_pattern=None,      
    lowercase=False,    
    stop_words=STOP_WORDS
)

/home/nicholas/.local/lib/python3.13/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/nicholas/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/nicholas/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Now that we have our vectorizer, let's train and cross-validate a simple Random Forest model with the default hyperparameters on it. For now, we'll ignore the keyword and location columns.

In [7]:
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

ct = make_column_transformer(
    (bow_vectorizer, "text"),
    ("drop", ["id", "keyword", "location"])
)
classifier = make_pipeline(ct, RandomForestClassifier(n_jobs=-1, random_state=123))

pd.DataFrame(cross_validate(classifier, X_train, y_train, cv=5, return_train_score=True)).T

,0,1,2,3,4
fit_time,4.998996,4.687885,3.971220,3.955560,4.042068
score_time,0.367120,0.458926,0.368594,0.361505,0.384569
test_score,0.718976,0.582403,0.604071,0.620894,0.710250
train_score,0.986207,0.985878,0.988506,0.987194,0.985388


As we can see from the cross-validation results, we're overfitting _a lot_, (the trees in the forest are currently permitted unlimited complexity) but that's fine, right now, we only want a model better than the dummy classifier. Let's go ahead and send it to Kaggle.

In [8]:
classifier.fit(X_train, y_train)
y_test = classifier.predict(df_test)

df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
df_result.set_index("id", inplace=True)
df_result.to_csv("1_randomforest.csv")

Uploading this to Kaggle gives us a score of 79.282% - Much better than the dummy model, but we could probably do better if we tuned the hyperparameters.

## Model 2: Simple bag-of-words with Neural Network

The classifier in use here could just as easily be swapped to any other model. Let's try an artificial neural network with the default parameters:

In [9]:
from sklearn.neural_network import MLPClassifier

ct = make_column_transformer(
    (bow_vectorizer, "text"),
    ("drop", ["id", "keyword", "location"])
)
classifier = make_pipeline(ct, MLPClassifier(random_state=123))

pd.DataFrame(cross_validate(classifier, X_train, y_train, cv=5, return_train_score=True)).T

,0,1,2,3,4
fit_time,63.879855,51.609148,57.009706,72.810667,58.758625
score_time,0.393656,0.441598,0.411306,0.418841,0.392033
test_score,0.674984,0.613920,0.629022,0.634034,0.697109
train_score,0.986043,0.985714,0.988342,0.986866,0.983911


Note that the default parameters here are overfitting as well, and it's also taking a _long_ time to fit. We'll need to do something to reduce the dimensionality of the vectorizer. Nonetheless, let's see what Kaggle thinks:

In [10]:
classifier.fit(X_train, y_train)
y_test = classifier.predict(df_test)

df_result = pd.DataFrame({"id": X_test["id"], "target": y_test})
df_result.set_index("id", inplace=True)
df_result.to_csv("2_neuralnetwork.csv")

Uploading this to Kaggle gives us a score of 76.463%, marginally worse than the Random Forest attempt. Note that there are several features and hyperparameters we're not taking advantage of, which we'll get into for the next few models.

## Model 3: Improved bag-of-words with Random Forest

There are several problems with the previous models. The first one that we want to tackle is that the "keyword" and "location" columns aren't being utilized. Let's fix that.

First, the keywords: an HTTP-escaped string denoting some especially important feature of the tweet. There are two obvious ways to handle this. The first is to simply one-hot encode the column. Let's count how many unique values there are:

In [11]:
len(X_train["keyword"].value_counts())

221

Hm. That'd be more features than some of our dimensionality reductions for the tweets - not ideal. The second method is to simply append the keyword to the tweet. This would have the effect of weighing it twice as much in the BOW encoding. That sounds good, but unfortunately, Scikit-Learn doesn't have a feature that lets us do this easily. So we'll define our own transformer:

In [12]:
import urllib

from sklearn.base import BaseEstimator, TransformerMixin

class KeywordJoiner(BaseEstimator, TransformerMixin):
    def __init__(self, unescape_cols=[]):
        self.unescape_cols = set(unescape_cols)
    
    def fit(self, X, y=None):
        pass
    
    def transform(self, X, y=None):
        final_series = pd.Series([""] * len(X), dtype=str)
        
        for colname, series in X.items():
            series = series.astype(str).fillna("")
            
            if colname in self.unescape_cols:
                series = series.apply(urllib.parse.unquote)
            
            final_series += " " + series
        
        return final_series.str.slice(start=1)

KeywordJoiner(["keyword"]).transform(X_train[["text", "keyword"]])

0       Our Deeds are the Reason of this #earthquake M...
1              Forest fire near La Ronge Sask. Canada nan
2       All residents asked to 'shelter in place' are ...
3       13,000 people receive #wildfires evacuation or...
4       Just got sent this photo from Ruby #Alaska as ...
                              ...                        
7608    Two giant cranes holding a bridge collapse int...
7609    @aria_ahrary @TheTawniest The out of control w...
7610    M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...
7611    Police investigating after an e-bike collided ...
7612    The Latest: More Homes Razed by Northern Calif...
Length: 7613, dtype: object

Now, for the location. It'd be nice if we could get rough latitude/longitude coordinates out of the city and country names given by Twitter users in their bio. Luckily, [Geopy](https://geopy.readthedocs.io/en/stable/) does just that by looking up an address/city/country via some online map backend (For this project, we'll use [Nominatim](https://nominatim.org/), an OpenStreetMap-based free online API). Let's make a transformer for this one too:

In [13]:
import gzip
import json

from geopy.geocoders import Nominatim

class GeopyLocator(BaseEstimator, TransformerMixin):
    def __init__(self, cachefile="geopy_cache.json.gz", backend=Nominatim):
        self.cachefile = cachefile
        
        try:
            with gzip.open(self.cachefile, "rt") as file:
                self.cache = json.load(file)
        except FileNotFoundError:
            self.cache = {}
            
        self.geolocator = backend(user_agent="ece_6420_final", timeout=10)
    
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        locations = []
        dirty = False
        
        try:
            for x in X:
                if x != x: # NaN
                    locations.append([np.nan, np.nan])
                elif x in self.cache:
                    if self.cache[x] is None:
                        locations.append([np.nan, np.nan])
                    else:
                        locations.append(self.cache[x])
                else:
                    print(f"Geolocating \"{x}\"...")
                    
                    if x.strip(): # Special case: geolocator crashes on all-whitespace strings
                        location = self.geolocator.geocode(x)
                    else:
                        location = None
                    dirty = True
                    time.sleep(1.5) # Respect Nominatim usage policy

                    if location is None:
                        self.cache[x] = None
                        locations.append([np.nan, np.nan])
                    else:
                        self.cache[x] = [location.latitude, location.longitude]
                        locations.append([location.latitude, location.longitude])
                        
            return np.array(locations)
        finally:
            if dirty:
                with gzip.open(self.cachefile, "wt") as file:
                    json.dump(self.cache, file)

GeopyLocator().transform(X_train["location"][:50])

array([[        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,         nan],
       [        nan,

Note that we are using a free, rate-limited API here - if we didn't fetch this information ahead-of-time, it would take **hours** to preprocess the data set. Instead of doing that, I recommend you just download our completed cache.

In [14]:
import hashlib

h = hashlib.md5()
md5sum = "447978d49bbf2d5c044be5a6f795d9c4"

with open("geopy_cache.json.gz", "rb+") as file:
    while (chunk := file.read(1 << 16)):
        h.update(chunk)

    if h.hexdigest() != md5sum:
        h = hashlib.md5()
        file.seek(0)
        
        with urllib.request.urlopen("https://raw.githubusercontent.com/thehashtableslasher/ece6420_final/refs/heads/main/geopy_cache.json.gz") as file2:
            while (chunk := file2.read(1 << 16)):
                file.write(chunk)
                h.update(chunk)

        file.truncate()
        
        assert h.hexdigest() == md5sum

Now that we've handled those two columns, it's time to move on to improving our text representation. Right now, we're using `CountVectorizer` with default parameters, so our feature set is maximized and our final dimensionality is huge. We can reduce the dimensionality with the `max_features` hyperparameter, and we _will_ use that later, but be can also reduce the dimensionality by mapping our vectors to a more compact feature space. PCA could be used, but latent semantic indexing (LSI) is often recommended for sparse feature spaces like BOW. <font color="red">NOTE: Connor and Daman, please test cross-validation between PCA and LSI</font>

In [15]:
# Daman: implement column transformer with above transformations and TruncatedSVD after CountVectorizer on text/keywords

Finally, we should actually tune our hyperparameters this time to reduce the complexity of the model. Let's perform a grid search <font color="red">NOTE: Connor, we can switch to RandomizedSearchCV if there are too many hyperparameters</font>

In [16]:
# Connor: implement GridSearchCV with RandomForestClassifier here. I recommend at least looking at the following:
# * max_features of CountVectorizer. Smallest value should be no less than the largest TruncatedSVD diensionality
# * n_components of TruncatedSVD. Default for Doc2Vec is 100, so at least use that.
# * n_estimators, criterion, and max_depth of RandomForestClassifier. For criterion, gini and entropy should both be 
#   tested.

Upload to Kaggle...

## Model 4: Improved bag-of-words with Neural Network

We can repeat the improvements we made to the Random Forest model with a neural network classifier as well:

In [17]:
# Connor: implement GridSearchCV with MLPClassifier here. I recommend at least looking at the following:
# * max_features of CountVectorizer. Same as before
# * n_components of TruncatedSVD. Same as before
# * hidden_layer_sizes of MLPClassifier. IMO, all hidden layers should be the same size, and the smallest layer size
#   value should be no less than the largest TruncatedSVD diensionality. You can do this easily with a list 
#   comprehension:
#   [[n] * m for n in [<your layer sizes here>] for m in [your layer counts here]]
# * activation of MLPClassifier. relu and logistic should both be tested.

Upload to Kaggle...

## Model 5: Doc2Vec with Random Forest

TODO: add description, https://cs.stanford.edu/~quocle/paragraph_vector.pdf https://radimrehurek.com/gensim/models/doc2vec.html

In [18]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

class D2VVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, **kwargs):
        self.model = None
        self.hyperparameters = kwargs
    
    def fit(self, X, y=None):
        self.model = Doc2Vec([TaggedDocument(tokenize(x), [i]) for i, x in enumerate(X)], **self.hyperparameters)
        return self

    def transform(self, X, y=None):
        return np.vstack([self.model.infer_vector(tokenize(x)) for x in X])

## Model 6: Doc2Vec with Neural Network

TODO: add description